# Within-sample nucleotide diversity
This independent report summarizes the existing per-sample microdiversity tables against the shared final vOTU catalogue. It does not require the abundance/lifestyle report or RNA enrichment. Pi describes observed read-level nucleotide diversity, not error-corrected viral population diversity. Compare callability alongside pi; missing calls are not zero diversity.


## Notebook setup, paths and plotting preferences


In [ ]:
from pathlib import Path
import base64
import html
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from Bio import SeqIO
from IPython.display import display

sns.set_style("ticks", {"axes.grid": True})
sns.set_palette("colorblind")
plt.rcParams.update({"font.family": "sans-serif", "font.sans-serif": ["Liberation Sans"],
                     "font.size": 9, "axes.linewidth": 0.5, "grid.linewidth": 0.25,
                     "svg.fonttype": "none", "savefig.dpi": 300, "figure.dpi": 150,
                     "text.usetex": False})
SAMPLES = list(snakemake.params.samples)
input_summaries = list(snakemake.input.summaries)
if len(SAMPLES) != len(set(SAMPLES)) or len(SAMPLES) != len(input_summaries):
    raise ValueError("Microdiversity requires one summary file per unique sample")
for path in snakemake.output:
    Path(path).parent.mkdir(parents=True, exist_ok=True)
figure_dir = Path(snakemake.output.figures)
figure_dir.mkdir(parents=True, exist_ok=True)
figure_files = []

def save_figure(fig, name):
    fig.tight_layout()
    png = figure_dir / (name + ".png")
    fig.savefig(png, bbox_inches="tight")
    fig.savefig(figure_dir / (name + ".svg"), bbox_inches="tight")
    figure_files.append(png)
    display(fig)
    plt.close(fig)


## Load the catalogue and sample results
Every summary must contain exactly the current catalogue. This prevents silently combining results from different catalogues or treating missing files as zero-diversity samples.


In [ ]:
with open(snakemake.input.fasta) as handle:
    catalogue = pd.DataFrame([[r.id, len(r.seq)] for r in SeqIO.parse(handle, "fasta")],
                             columns=["contig", "catalogue_length"])
if catalogue["contig"].duplicated().any():
    raise ValueError("Duplicate identifiers in the final catalogue")
ids = pd.Index(catalogue["contig"], name="contig")
lengths = catalogue.set_index("contig")["catalogue_length"]
columns = ["sample", "contig", "length", "acgt_reference_sites", "covered_sites", "callable_sites",
           "callable_fraction", "depth_capped_sites", "mean_depth", "polymorphic_sites",
           "pi_callable", "entropy_callable_bits", "min_depth", "min_base_quality",
           "min_mapping_quality", "min_alt_count", "min_alt_frequency", "max_depth"]
frames = []
for sample, path in zip(SAMPLES, input_summaries):
    frame = pd.read_csv(path, sep="\t", dtype={"sample": str, "contig": str})
    if not set(columns).issubset(frame):
        raise ValueError(f"Missing microdiversity columns in {path}")
    if frame["contig"].duplicated().any() or set(frame["contig"]) != set(ids):
        raise ValueError(f"Catalogue identifiers differ in {path}")
    if not frame["sample"].eq(sample).all():
        raise ValueError(f"Sample identifiers differ in {path}")
    for column in columns[2:]:
        frame[column] = pd.to_numeric(frame[column], errors="raise")
    if not frame.set_index("contig")["length"].reindex(ids).eq(lengths).all():
        raise ValueError(f"Catalogue lengths differ in {path}")
    if frame["callable_sites"].isna().any() or (frame["callable_sites"] < 0).any() or (frame["callable_sites"] > frame["acgt_reference_sites"]).any():
        raise ValueError(f"Invalid callable-site counts in {path}")
    callable_rows = frame["callable_sites"].gt(0)
    if frame.loc[callable_rows, ["pi_callable", "entropy_callable_bits"]].isna().any().any():
        raise ValueError(f"Callable sites without diversity estimates in {path}")
    frame.loc[~callable_rows, ["pi_callable", "entropy_callable_bits"]] = np.nan
    frames.append(frame)
combined = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=columns)
combined["callability_status"] = np.where(combined["callable_sites"].gt(0), "Callable sites available", "No callable sites")
combined.to_csv(snakemake.output.combined, index=False)
pi_matrix = combined.pivot(index="contig", columns="sample", values="pi_callable").reindex(index=ids, columns=SAMPLES)
callable_matrix = combined.pivot(index="contig", columns="sample", values="callable_fraction").reindex(index=ids, columns=SAMPLES)
pi_matrix.to_csv(snakemake.output.pi)
callable_matrix.to_csv(snakemake.output.callable_fraction)
display(combined.head())


## Per-sample summaries
Diversity and entropy are weighted by the number of callable sites, including invariant sites. Whole-catalogue mean depth is weighted by sequence length. A sample without callable sites has NA diversity, not zero.


In [ ]:
sample_rows = []
for sample in SAMPLES:
    frame = combined.loc[combined["sample"].eq(sample)]
    called = frame["callable_sites"].gt(0)
    sites = frame.loc[called, "callable_sites"].sum()
    acgt_sites = frame["acgt_reference_sites"].sum()
    total_length = frame["length"].sum()
    sample_rows.append([sample, len(frame), int(frame["covered_sites"].gt(0).sum()), int(called.sum()),
                        sites, sites / acgt_sites if acgt_sites else np.nan,
                        frame["depth_capped_sites"].sum(), frame["polymorphic_sites"].sum(),
                        (frame.loc[called, "pi_callable"] * frame.loc[called, "callable_sites"]).sum() / sites if sites else np.nan,
                        (frame.loc[called, "entropy_callable_bits"] * frame.loc[called, "callable_sites"]).sum() / sites if sites else np.nan,
                        (frame["mean_depth"] * frame["length"]).sum() / total_length if total_length else np.nan,
                        "Callable sites available" if sites else "No callable sites"])
sample_summary = pd.DataFrame(sample_rows, columns=["sample", "catalogue_vOTUs", "covered_vOTUs", "callable_vOTUs",
    "callable_sites", "catalogue_callable_fraction", "depth_capped_sites", "polymorphic_sites",
    "pi_callable_site_weighted", "entropy_callable_site_weighted_bits", "catalogue_mean_depth", "status"])
sample_summary.to_csv(snakemake.output.samples, index=False)
display(sample_summary)


## Coverage, callability and observed diversity


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
if SAMPLES:
    indexed_samples = sample_summary.set_index("sample")
    indexed_samples["callable_vOTUs"].plot.bar(ax=axes[0])
    indexed_samples["catalogue_callable_fraction"].plot.bar(ax=axes[1])
    pi_values = indexed_samples["pi_callable_site_weighted"]
    if pi_values.notna().any():
        pi_values.plot.bar(ax=axes[2])
    else:
        axes[2].text(0.5, 0.5, "No callable sites", ha="center", transform=axes[2].transAxes)
axes[0].set_title("vOTUs with callable sites")
axes[1].set_title("Callable fraction of A/C/G/T reference sites")
axes[1].set_ylim(0, 1)
axes[2].set_title("Callable-site-weighted pi")
save_figure(fig, "sample_microdiversity")

fig, axes = plt.subplots(1, 2, figsize=(12, 8))
# Rank on callability, not on pi, to avoid selecting only highly variable vOTUs.
top = callable_matrix.mean(axis=1).sort_values(ascending=False).head(30).index
for ax, matrix, label in [(axes[0], callable_matrix, "Callable fraction"), (axes[1], pi_matrix, "Pi at callable sites")]:
    data = matrix.reindex(top)
    if len(top) and SAMPLES and data.notna().any().any():
        sns.heatmap(data, ax=ax, cmap="viridis", mask=data.isna(), cbar_kws={"label": label})
    else:
        ax.text(0.5, 0.5, "No defined values", ha="center", transform=ax.transAxes)
    ax.set_title(label + " — top 30 by callability")
save_figure(fig, "callability_pi_heatmaps")

fig, ax = plt.subplots(figsize=(6, 4))
for sample in SAMPLES:
    frame = combined.loc[combined["sample"].eq(sample) & combined["callable_sites"].gt(0)]
    ax.scatter(frame["callable_fraction"], frame["pi_callable"], s=15, alpha=0.7, label=sample)
ax.set(xlabel="Callable fraction", ylabel="Pi at callable sites", xlim=(0, 1))
if 0 < len(SAMPLES) <= 15:
    ax.legend()
save_figure(fig, "pi_vs_callability")


## Export report and interpretation notes


In [ ]:
threshold_columns = ["sample", "min_depth", "min_base_quality", "min_mapping_quality",
                     "min_alt_count", "min_alt_frequency", "max_depth"]
thresholds = combined[threshold_columns].drop_duplicates()
sections = ["<html><head><meta charset='utf-8'><title>Microdiversity summary</title></head><body>",
            "<h1>Within-sample nucleotide diversity</h1>",
            "<p>Independent of RNA enrichment and the abundance/lifestyle report. Results refer to the shared filtered catalogue. "
            "Pi includes invariant callable sites. Missing calls remain NA. Capped sites are excluded by the underlying analysis. "
            "Read-level variation may include sequencing, RT/PCR errors and cross-mapping; these are not validated population variants "
            "or evidence of viral activity. No automatic cross-library significance tests are performed.</p>",
            "<h2>Sample summary</h2>" + sample_summary.to_html(index=False),
            "<h2>Recorded analysis thresholds</h2>" + thresholds.to_html(index=False)]
for name, path in snakemake.output.items():
    if name not in ["html", "figures"]:
        relative_path = Path(path).relative_to(Path(snakemake.output.html).parent)
        sections.append('<p><a href="' + html.escape(str(relative_path), quote=True) + '">' + html.escape(name) + "</a></p>")
for path in figure_files:
    encoded = base64.b64encode(path.read_bytes()).decode()
    sections.append("<h2>" + html.escape(path.stem.replace("_", " ")) + "</h2><img style='max-width:100%' src='data:image/png;base64," + encoded + "'>")
sections.append("</body></html>")
Path(snakemake.output.html).write_text("\n".join(sections), encoding="utf-8")
print("Report:", snakemake.output.html)
